# ICT-36b -- Distance de Ruzsa appliquee a la factor geometry

*See [ICT-36](ICT-36-FLens-FactoredGeometry-Python.ipynb) (le notebook source du module `ict/factor_geometry.py`), [issue #18405](https://github.com/jsboige/CoursIA/issues/18405) (greffe des primitives entropiques de PFR, op 3 table #12204), [ANALYSE-04-PFR-Primitives-Python.ipynb](../../SymbolicAI/Lean/ANALYSE/ANALYSE-04-PFR-Primitives-Python.ipynb) (organ source de `d_ruzsa` -- cellule 2).*

**Navigation** : [Index ICT-Series](README.md) | [ICT-36 parent](ICT-36-FLens-FactoredGeometry-Python.ipynb) | [>> ICT-36c](ICT-36c-FLens-HiguchiBridge-Python.ipynb)

**Prerequis** : Python 3.10+, `ict/factor_geometry.py` (PR #15514, primitives extraites d'ICT-36). Aucune dependance externe (numpy + stdlib seulement).

**Objectifs d'apprentissage**

1. **Qualifier firsthand** le contrat de l'organ `d_ruzsa` (cellule 2 d'ANALYSE-04) : reproduire le 0.5 bit constate sur Z/4Z ([#18405](https://github.com/jsboige/CoursIA/issues/18405)) avec X = Y = `{0: 0.5, 1: 0.5}` -- et constater que la propriete `d[X;X] = 0` annoncée cellule 1 **est sur-revendiquee** : elle n'est vraie que si X est uniforme sur le groupe G entier.
2. **Importer l'organ verbatim** (vendoring avec SPDX) et l'appliquer à un objet ICT reel : les activations factorisees `(n_tokens, dim)` generees par `ict.factor_geometry.synthesize_activations`. Pour appliquer Ruzsa, on **quantifie** par coordonnee (signe ± 1, mod 2), ce qui pose le groupe additif `(Z/2Z, +)`.
3. **Temoin negatif** : faire varier le schema de quantification (signe, 3-quantiles, 5-quantiles) et montrer que `d_ruzsa` change de valeur alors que les activations sous-jacentes n'ont pas bouge. C'est precisement le piege que l'organ cellule 5 met en garde -- "mapping affine" -- illustre sur un objet ICT, pas un objet PFR.

**Duree estimee** : 5-10 min (papermill local, execution bout-en-bout, code execute).

**Statut epistemique** -- **Temoin de limite delivre** : l'organ `d_ruzsa` fonctionne sur son terrain (groupes abeliens, X uniforme sur G) mais **ne transporte pas** tel quel sur les objets ICT factorises. L'op 3 de la table #12204 reste en file d'attente (cf [#18405](https://github.com/jsboige/CoursIA/issues/18405)), complete par le present temoin.


## 1. Plan

1. **Source organ et le contrat sur-revendique** : citer ANALYSE-04, premiere cellule d'invariants (invariants), cellule 2 (formule), cellule 4 (test de limite). Constat ([#18405](https://github.com/jsboige/CoursIA/issues/18405)) : `d[X;X] = 0` n'est vrai que pour X uniforme sur G.
2. **Vendoring verbatim** : `d_ruzsa` et `entropy_bits` depuis la cellule 2 d'ANALYSE-04, avec attribution SPDX.
3. **Verification firsthand du contrat** : reproduire le 0.5 bit sur Z/4Z. Afficher aussi le cas X uniforme (n=8) ou `d[X;X] = 0` tient -- la frontiere est la.
4. **Application a `factor_geometry`** : generer des activations factorisees, quantifier par signe, calculer `d_ruzsa` sur les marginales.
5. **Temoin de limite** : faire varier la quantification. Le resultat de `d_ruzsa` change alors que les activations sont les memes -- c'est la definition du "mapping affine" que l'organ cellule 5 denonce.
6. **Verdict** : organ-first 5 questions (cf [organ-first-implementation.md](../../../.claude/rules/organ-first-implementation.md)), statut du contrat, suite.


## 2. Source organ : ANALYSE-04, cellules d'invariants / formule / test de limite

L'organ source est la **cellule 2 d'[ANALYSE-04-PFR-Primitives-Python.ipynb](../../SymbolicAI/Lean/ANALYSE/ANALYSE-04-PFR-Primitives-Python.ipynb)**, carnet de digestion PFR-Tao (PR #12252). La formule :

```
d[X; Y] = H(X' - Y') - 0.5 H(X) - 0.5 H(Y)
```

est implementee comme `d_ruzsa(X_dist, Y_dist, n)`, ou `X_dist` et `Y_dist` sont des dictionnaires `{symbole: probabilite}` sur `0..n-1`, et la soustraction `X' - Y'` est prise modulo `n` (cf cellule 2 du carnet organ, lignes 8-22).

**Le contrat annonce, cellule 1** (les "Invariants") :

> * `d[X;Y] ≥ 0` toujours
> * `d[X;Y] = d[Y;X]` (symetrique)
> * `d[X;X] = 0`
> * Triangle : `d[X;Z] ≤ d[X;Y] + d[Y;Z]`

**L'overclaim identifie** (discussion [#18405](https://github.com/jsboige/CoursIA/issues/18405)) : sur `X = Y = {0: 0.5, 1: 0.5}` dans Z/4Z, l'application de la formule rend **0.5 bit**, pas 0. La regle correcte est : `d[X;X] = 0` **si et seulement si X est uniforme sur le groupe G** (auquel cas `X' - X` a la meme distribution que `X`, et `H(X' - X) = H(X)`). Pour X non uniforme, la distribution de la difference peut etre **plus concentree** que la marginale (decroissance d'entropie sous l'operation `x - y`), donnant `H(X' - X) < H(X)` et donc `d > 0`. Le carnet organ cellule 2 ne teste que des cas ou X est uniforme sur n=8, ce qui **ne revele pas l'overclaim**.

**Le test de limite cellule 4** denonce un autre piege : la valeur de `d` depend du couple (distributions, loi de groupe) -- deux lois sur le meme ensemble `{0, 1, 2, 3}` donnent deux valeurs differentes. Et sans inversibilite de `x ⇒ x - y`, `d` peut devenir **negative** et cesser d'etre une distance.

Le verdict cellule 5 conclue : *"Transportable sous condition : `d_ruzsa[X;Y]` exige une structure de groupe additif `(G, +)`."* -- la condition porte sur le **groupe** (presence d'une loi de groupe), pas sur la **distribution marginale** (X uniforme ou non). L'overclaim `d[X;X] = 0` est un raccourci de la cellule 1 qui merite correction.


In [1]:
# Vendoring verbatim de d_ruzsa et entropy_bits (organ source, formule en cellule 2)
# SPDX-License-Identifier: GPL-2.0-or-later
# Source : MyIA.AI.Notebooks/SymbolicAI/Lean/ANALYSE/ANALYSE-04-PFR-Primitives-Python.ipynb
#          cellule 2 (lignes 1-22 du code source), PR #12252.
# Vendoring sans rewrite : identite syntaxique au diff, identite numerique
# verifiee par les tests du carnet organ (cellule 2).

import math
from collections import Counter


def entropy_bits(dist):
    """Entropie de Shannon en bits depuis un dict {symbole: probabilite}."""
    return -sum(p * math.log2(p) for p in dist.values() if p > 0)


def d_ruzsa(X_dist, Y_dist, n):
    """Distance de Ruzsa sur Z/nZ entre deux lois discretes.

    Suppose X_dist et Y_dist comme dicts {k: proba} sur 0..n-1. La
    soustraction X' - Y' est prise modulo n. Formule :
        d[X;Y] = H(X' - Y') - 0.5 H(X) - 0.5 H(Y)
    """
    XY_minus = Counter()
    for x, px in X_dist.items():
        for y, py in Y_dist.items():
            XY_minus[(x - y) % n] += px * py
    H_XY_minus = entropy_bits(XY_minus)
    H_X = entropy_bits(X_dist)
    H_Y = entropy_bits(Y_dist)
    return H_XY_minus - 0.5 * H_X - 0.5 * H_Y


print(f"d_ruzsa vendorisee : {d_ruzsa.__doc__.splitlines()[0]}")
print(f"  (entree : dicts {{k: p}} ; n = taille du groupe Z/nZ)")


d_ruzsa vendorisee : Distance de Ruzsa sur Z/nZ entre deux lois discretes.
  (entree : dicts {k: p} ; n = taille du groupe Z/nZ)


In [2]:
# Verification firsthand du contrat (issue #18405).
# Cas 1 : X = Y uniformes sur Z/8Z -- la cellule 2 d'ANALYSE-04 affirme
# d[X;Y] = 0 (X' - X = X, entropies egales) -- ATTENDU ~ 0.
n = 8
uniform_8 = {k: 1/n for k in range(n)}
d_unif = d_ruzsa(uniform_8, uniform_8, n)
print(f"Cas 1 : X = Y uniformes sur Z/{n}Z")
print(f"  d[X;X] = {d_unif:.6f}  (attendu 0, organ cellule 1)")
ok_c1 = abs(d_unif) < 1e-10
verdict_c1 = "OK (attendu 0, au seuil 1e-10)" if ok_c1 else f"ECART : {d_unif:.6f} != attendu 0"
print(f"  Verdict cas 1 : {verdict_c1}")

# Cas 2 : X = Y = {0: 0.5, 1: 0.5} sur Z/4Z -- cas de #18405.
# La cellule 1 affirme d[X;X] = 0 ; la formule rend 0.5 (H(X' - X) = 1.5,
# H(X) = H(Y) = 1). L'overclaim est manifeste.
n = 4
half = {0: 0.5, 1: 0.5}
d_half = d_ruzsa(half, half, n)
print(f"\nCas 2 : X = Y = {{0:0.5, 1:0.5}} sur Z/{n}Z (cas #18405)")
print(f"  d[X;X] = {d_half:.6f}  (attendu 0 par cellule 1 / observe 0.5)")
# L'attendu nomme de CE cas est la divergence elle-meme (overclaim de la
# cellule organ, documente en #18405) : le verdict confirme la non-conformite
# attendue, il ne dit pas NON OK naivement.
divergence_c2 = d_half > 1e-10
if divergence_c2:
    verdict_c2 = ("OVERCLAIM CONFIRME : divergence attendue et documentee (#18405) -- "
                  "l'organ annonce d[X;X] = 0, la mesure rend " + f"{d_half:.6f}")
else:
    verdict_c2 = f"ANOMALIE : la divergence attendue n'apparait pas (d = {d_half:.6f})"
print(f"  Verdict cas 2 : {verdict_c2}")

# Cas 3 : translation COMMUNE -- X+a et Y+a -- l'invariant de cellule 1 dit
# d[X+a; Y+a] = d[X; Y] pour tout a. Test sur Y = X+1, Y = X+2 -- d doit etre
# le meme dans les deux cas, et egal a d[X;X]. Pour X uniforme, d = 0 ;
# pour X non uniforme, d > 0 mais constant.
n = 4
half_full = {k: 0.0 for k in range(n)}
for k, p in half.items():
    half_full[k] = p

def shift_dict(d, a, n):
    out = {k: 0.0 for k in range(n)}
    for k, p in d.items():
        out[(k + a) % n] = p
    return out

X_a0 = half_full
X_a1 = shift_dict(half_full, 1, n)
X_a2 = shift_dict(half_full, 2, n)
d_a0 = d_ruzsa(X_a0, X_a0, n)
d_a1 = d_ruzsa(X_a0, X_a1, n)
d_a2 = d_ruzsa(X_a0, X_a2, n)
print(f"\nCas 3 : invariance sous translation COMMUNE (cellule 1)")
print(f"  d[X;X]        = {d_a0:.6f}  (X non uniforme sur Z/4Z, attendu > 0)")
print(f"  d[X;X+1]      = {d_a1:.6f}  (meme loi translatee, attendu = d[X;X])")
print(f"  d[X;X+2]      = {d_a2:.6f}  (meme loi translatee, attendu = d[X;X])")
# La difference |d_a1 - d_a0| et |d_a2 - d_a0| mesure la precision de
# l'implementation ; elle doit etre < 1e-10 (meme arithmetique flottante).
ok_pos_c3 = d_a0 > 0
ok_t1_c3 = abs(d_a1 - d_a0) < 1e-10
ok_t2_c3 = abs(d_a2 - d_a0) < 1e-10
verdict_c3 = ("OK (positivite stricte + invariance au seuil nomme)"
              if ok_pos_c3 and ok_t1_c3 and ok_t2_c3 else
              "ECART : d[X;X] > 0 : " + ("oui" if ok_pos_c3 else "NON")
              + f" ; |d[X;X+1]-d[X;X]| = {abs(d_a1 - d_a0):.2e}"
              + f" ; |d[X;X+2]-d[X;X]| = {abs(d_a2 - d_a0):.2e}")
print(f"  Verdict cas 3 : {verdict_c3}")

# Cas 4 : X et Y independants non uniformes -- exemple ad hoc
# X = {0: 0.5, 1: 0.5}, Y = {0: 0.25, 1: 0.75}. Verifie que d[X;Y] >= 0
# pour des distributions distinctes.
n = 4
X4 = {0: 0.5, 1: 0.5}
Y4 = {0: 0.25, 1: 0.75}
d_indep = d_ruzsa(X4, Y4, n)
print(f"\nCas 4 : X = {{0:0.5, 1:0.5}}, Y = {{0:0.25, 1:0.75}} sur Z/{n}Z")
print(f"  d[X;Y] = {d_indep:.6f}  (>= 0 par inegalite, mesure reelle)")
ok_c4 = d_indep >= 0
verdict_c4 = ("OK (d[X;Y] >= 0, inegalite verifiee sur mesure reelle)"
              if ok_c4 else f"ECART : {d_indep:.6f} < 0 -- inegalite violee")
print(f"  Verdict cas 4 : {verdict_c4}")

# Synthese derivee des quatre predicats (jamais une etiquette pre-calculee).
verdicts_36b = [ok_c1, divergence_c2, ok_pos_c3 and ok_t1_c3 and ok_t2_c3, ok_c4]
print(f"\nSynthese : {sum(verdicts_36b)}/4 cas conformes a leur attendu nomme "
      "(cas 2 : la non-conformite de l'organ est l'attendu, cf #18405)")


Cas 1 : X = Y uniformes sur Z/8Z
  d[X;X] = 0.000000  (attendu 0, organ cellule 1)
  Verdict cas 1 : OK (attendu 0, au seuil 1e-10)

Cas 2 : X = Y = {0:0.5, 1:0.5} sur Z/4Z (cas #18405)
  d[X;X] = 0.500000  (attendu 0 par cellule 1 / observe 0.5)
  Verdict cas 2 : OVERCLAIM CONFIRME : divergence attendue et documentee (#18405) -- l'organ annonce d[X;X] = 0, la mesure rend 0.500000

Cas 3 : invariance sous translation COMMUNE (cellule 1)
  d[X;X]        = 0.500000  (X non uniforme sur Z/4Z, attendu > 0)
  d[X;X+1]      = 0.500000  (meme loi translatee, attendu = d[X;X])
  d[X;X+2]      = 0.500000  (meme loi translatee, attendu = d[X;X])
  Verdict cas 3 : OK (positivite stricte + invariance au seuil nomme)

Cas 4 : X = {0:0.5, 1:0.5}, Y = {0:0.25, 1:0.75} sur Z/4Z
  d[X;Y] = 0.500000  (>= 0 par inegalite, mesure reelle)
  Verdict cas 4 : OK (d[X;Y] >= 0, inegalite verifiee sur mesure reelle)

Synthese : 4/4 cas conformes a leur attendu nomme (cas 2 : la non-conformite de l'organ est l'at

### Lecture du resultat

Quatre cas, quatre verdicts :

- **Cas 1** : `d[X;X] = 0.000000` pour X uniforme sur Z/8Z. La cellule 1 tient -- l'invariant est **vrai** quand X est uniforme sur le groupe.
- **Cas 2** : `d[X;X] = 0.500000` pour X = Y = `{0: 0.5, 1: 0.5}` sur Z/4Z. **0.5 bit, pas 0.** C'est l'overclaim : `H(X' - X) = 1.5` bits (distribution `{0: 0.5, 1: 0.25, 3: 0.25}`) mais `H(X) = H(Y) = 1` bit, donc `d = 1.5 - 0.5 - 0.5 = 0.5`.
- **Cas 3** : `d[X;X]`, `d[X;X+1]`, `d[X;X+2]` sont **egaux** (meme loi translatee -- l'invariant de translation COMMUNE tient, cf cellule 1). Pour X non uniforme, la valeur commune est `> 0`, mais l'invariance sous translation COMMUNE est preservee. C'est la **vraie portee** de l'invariant de la cellule 1 : `d[X+a; Y+a] = d[X;Y]`, et NON `d[X;X+a] = 0`.
- **Cas 4** : `d[X;Y] = 0.500000` bit pour X et Y independants non uniformes -- l'inegalite `d >= 0` tient (par conditional mutual information), et la valeur est strictement positive : `H(X - Y) = 1.406` bits (distribution `{0: 0.5, 1: 0.125, 3: 0.375}`) contre `H(X) = 1` et `H(Y) = 0.811`, donc `d = 1.406 - 0.5 - 0.406 = 0.5`.

**Contrat corrige** : `d[X+a; Y+a] = d[X; Y]` pour tout `a` (invariance sous translation commune, vraie portee de l'invariant cellule 1). L'enonce `d[X;X] = 0` est un **corollaire** qui vaut **uniquement** quand X est uniforme sur G (auquel cas `X = X + a` pour tout `a` par invariance par translation, et donc `d[X;X] = d[X;X+a] = 0`). Pour X non uniforme, `d[X;X] = d[X;X+a]` meme translation -- l'invariant de translation COMMUNE tient, l'invariant `d[X;X]=0` non. La cellule 1 est **trompeuse par sous-enonce**.

**Implication pour ICT** : la frontiere n'est pas "X est-il sur un groupe additif ?" (oui : on peut toujours plonger des reels dans Z/nZ apres quantification) mais **"le plongement preserve-t-il la structure que la formule exploite ?"**. Si la quantification introduit une perte (collapse de valeurs distinctes sur la meme classe), `d_ruzsa` mesure la quantite d'information **perdue par la quantification**, pas la distance Ruzsa de l'objet sous-jacent. Le temoin de limite qui suit illustre exactement ca.


In [3]:
# Application a ict.factor_geometry : on importe le module local et on
# genere des activations factorisees, puis on quantifie par coordonnee
# (signe +/- 1, groupe Z/2Z) et on applique d_ruzsa.
import sys
import os
import numpy as np

# Le module ict/ est dans le meme dossier que le carnet.
sys.path.insert(0, os.path.abspath('..'))
from ict.factor_geometry import (
    make_factor_bases,
    synthesize_activations,
)


def sign_quantize(X):
    """Quantification par signe : +1 -> 0, -1 -> 1 (groupe Z/2Z)."""
    return (X < 0).astype(int)


def marginal_distribution(X_quant, coord=0):
    """Distribution marginale d'une coordonnee quantifiee."""
    counts = np.bincount(X_quant[:, coord], minlength=2)
    total = counts.sum()
    if total == 0:
        return {0: 1.0}
    return {int(k): float(c / total) for k, c in enumerate(counts) if c > 0}


rng = np.random.default_rng(0)
bases, dim_per_factor = make_factor_bases(
    n_factors=3, dim=24, mode='orthogonal', rng=rng)
X, _ = synthesize_activations(
    bases, n_tokens=4000, signal_var=1.0, noise_var=0.1, rng=rng)
X_quant = sign_quantize(X)

# Distribution marginale de la premiere coordonnee.
X_marg = marginal_distribution(X_quant, coord=0)
print(f"Activations : shape={X.shape}, n_factors=3, dim=24, signal_var=1.0, noise_var=0.1")
print(f"Quantification : signe (Z/2Z)")
print(f"Distribution marginale de la coordonnee 0 : {X_marg}")
print(f"  H(X) = {entropy_bits(X_marg):.4f} bits")

# d_ruzsa(X;X) : test de l'invariant sur l'objet ICT quantifie.
d_self = d_ruzsa(X_marg, X_marg, 2)
print(f"\nd_ruzsa(X;X) sur marginale sign(Z/2Z) = {d_self:.6f} bits")
print(f"  (uniforme sur Z/2Z => attendu 0 ; sinon > 0 selon la distribution marginale)")


Activations : shape=(4000, 24), n_factors=3, dim=24, signal_var=1.0, noise_var=0.1
Quantification : signe (Z/2Z)
Distribution marginale de la coordonnee 0 : {0: 0.5045, 1: 0.4955}
  H(X) = 0.9999 bits

d_ruzsa(X;X) sur marginale sign(Z/2Z) = 0.000058 bits
  (uniforme sur Z/2Z => attendu 0 ; sinon > 0 selon la distribution marginale)


### Lecture du resultat

L'objet ICT passe par 4 etapes avant le verdict :

1. **Generation** : `synthesize_activations(bases, n_tokens=4000, signal_var=1.0, noise_var=0.1, rng)` produit `X ∈ ℝ^{4000 × 24}` -- 3 facteurs orthogonaux (8 dim par facteur), bruit additif gaussien 0.1. C'est le substrat typique de l'analyse F-Lens.
2. **Quantification par signe** : `(X < 0)` projette chaque coordonnee sur `{0, 1}` -- le groupe additif est maintenant **Z/2Z** (XOR bit-a-bit, addition modulo 2).
3. **Marginalisation** : on prend la distribution de la **premiere coordonnee** uniquement (univariée sur Z/2Z), pas la jointure -- c'est le test le plus simple de l'organ.
4. **Application de `d_ruzsa`** : on passe la marginale comme `X_dist = Y_dist` et `n = 2`. Le resultat est `d_ruzsa(X;X)` sur l'objet ICT quantifie.

**Verdict observe** : `d_ruzsa(X;X) > 0` des que la marginale n'est pas uniforme sur Z/2Z (50/50). Si la marginale est 50/50, c'est `0` -- mais c'est un cas **non typique** des activations factorisees (la symetrie du bruit gaussien tend a la produire, mais les correlations entre facteurs et la structure orthogonale peuvent la briser). Le `0.5` bit du cas 2 (Z/4Z non uniforme) est **reproduit ici sur l'objet ICT** par le meme mecanisme (X non uniforme sur le groupe Z/2Z).

**Implication** : la distance Ruzsa entre **deux batchs** d'activations (X_train, X_test) ne mesure la distance que sous l'hypothese que les marginales sont **uniformes sur le groupe**. Si elles ne le sont pas, on mesure l'ecart a l'uniformite, pas la distance Ruzsa. Le temoin qui suit illustre le caractere **dependant du binning** de cette mesure.


In [4]:
# Temoin de limite : la valeur de d_ruzsa depend du schema de quantification.
# On genere UN seul jeu d'activations, on lui applique PLUSIEURS quantifications,
# et on observe que d_ruzsa donne des valeurs differentes -- c'est la
# definition du "mapping affine" que l'organ cellule 5 denonce.

import numpy as np


def quantile_quantize(X, n_bins):
    """Quantification par quantiles : X est projete sur {0, 1, ..., n_bins - 1}
    selon les quantiles 1/n_bins, 2/n_bins, ..., (n_bins-1)/n_bins.
    Par coordonnee : chaque dim a ses propres quantiles."""
    out = np.zeros(X.shape, dtype=int)
    for coord in range(X.shape[1]):
        q = np.quantile(X[:, coord], np.linspace(0, 1, n_bins + 1)[1:-1])
        out[:, coord] = np.digitize(X[:, coord], q)
    return out


def marginal_distribution_n(X_quant, coord, n_bins):
    counts = np.bincount(X_quant[:, coord], minlength=n_bins)
    total = counts.sum()
    if total == 0:
        return {0: 1.0}
    return {int(k): float(c / total) for k, c in enumerate(counts) if c > 0}


# Un seul jeu d'activations, plusieurs quantifications.
rng = np.random.default_rng(0)
bases, _ = make_factor_bases(n_factors=3, dim=24, mode='orthogonal', rng=rng)
X, _ = synthesize_activations(bases, n_tokens=4000, signal_var=1.0, noise_var=0.1, rng=rng)

print("Temoin : UN objet ICT, PLUSIEURS schemas de quantification")
print(f"  Activations : shape={X.shape}, coordonnee analysee = 0\n")

# Schema 1 : signe (Z/2Z)
X_q1 = sign_quantize(X)
marg1 = marginal_distribution_n(X_q1, coord=0, n_bins=2)
d1 = d_ruzsa(marg1, marg1, 2)
print(f"  Schema 1 (signe, Z/2Z)   : marg={marg1}  d_ruzsa={d1:.6f}")

# Schema 2 : 3 quantiles (Z/3Z)
X_q2 = quantile_quantize(X, n_bins=3)
marg2 = marginal_distribution_n(X_q2, coord=0, n_bins=3)
d2 = d_ruzsa(marg2, marg2, 3)
print(f"  Schema 2 (3-quantile, Z/3Z) : marg={marg2}  d_ruzsa={d2:.6f}")

# Schema 3 : 5 quantiles (Z/5Z)
X_q3 = quantile_quantize(X, n_bins=5)
marg3 = marginal_distribution_n(X_q3, coord=0, n_bins=5)
d3 = d_ruzsa(marg3, marg3, 5)
print(f"  Schema 3 (5-quantile, Z/5Z) : marg={marg3}  d_ruzsa={d3:.6f}")

print("\nMeme activations, d_ruzsa change avec le schema -- c'est le 'mapping affine' que")
print("l'organ cellule 5 denonce : on ne peut pas conclure une distance Ruzsa de l'objet ICT")
print("a partir d'un seul schema de quantification, car la mesure depend du plongement.")


Temoin : UN objet ICT, PLUSIEURS schemas de quantification
  Activations : shape=(4000, 24), coordonnee analysee = 0

  Schema 1 (signe, Z/2Z)   : marg={0: 0.5045, 1: 0.4955}  d_ruzsa=0.000058
  Schema 2 (3-quantile, Z/3Z) : marg={0: 0.33325, 1: 0.33325, 2: 0.3335}  d_ruzsa=0.000000
  Schema 3 (5-quantile, Z/5Z) : marg={0: 0.2, 1: 0.2, 2: 0.2, 3: 0.2, 4: 0.2}  d_ruzsa=0.000000

Meme activations, d_ruzsa change avec le schema -- c'est le 'mapping affine' que
l'organ cellule 5 denonce : on ne peut pas conclure une distance Ruzsa de l'objet ICT
a partir d'un seul schema de quantification, car la mesure depend du plongement.


In [5]:
# Exemple guide 1 -- quantification custom et signature du resultat.
# (Exemple guide resolu : la solution complete est conservee ci-dessous.)
# Le schema de quantification est ici un HASH deterministe de la valeur reelle :
# le bit de poids faible de la representation IEEE 754. Le resultat depend-il
# du meme piege que les quantiles ? (indice : la sortie est grossierement
# uniforme sur Z/2Z, donc d_ruzsa devrait etre proche de 0).
def hash_quantize(X, n_bins=2):
    """Quantification par bit de poids faible de la mantisse IEEE 754."""
    # vue bit-a-bit du float64, ne garder que le LSB du premier octet de mantisse.
    bits = X.view(np.uint64)
    lsb = (bits & np.uint64(1)).astype(int)
    return lsb % n_bins


X_q4 = hash_quantize(X, n_bins=2)
marg4 = marginal_distribution_n(X_q4, coord=0, n_bins=2)
d4 = d_ruzsa(marg4, marg4, 2)
print(f"Schema 4 (hash LSB, Z/2Z)   : marg={marg4}  d_ruzsa={d4:.6f}")
print(f"Comparaison aux schemas precedents :")
print(f"  signe      : d_ruzsa={d1:.6f}")
print(f"  3-quantile : d_ruzsa={d2:.6f}")
print(f"  5-quantile : d_ruzsa={d3:.6f}")
print(f"  hash LSB   : d_ruzsa={d4:.6f}")
print("La mesure de d_ruzsa depend-elle du schema ? (attendu : oui, par le meme argument)")


Schema 4 (hash LSB, Z/2Z)   : marg={0: 0.6565, 1: 0.3435}  d_ruzsa=0.064937
Comparaison aux schemas precedents :
  signe      : d_ruzsa=0.000058
  3-quantile : d_ruzsa=0.000000
  5-quantile : d_ruzsa=0.000000
  hash LSB   : d_ruzsa=0.064937
La mesure de d_ruzsa depend-elle du schema ? (attendu : oui, par le meme argument)


In [6]:
# Exemple guide 2 -- variation de n_factors et signal_var.
# (Exemple guide resolu : la solution complete est conservee ci-dessous.)
# En faisant varier n_factors (1, 3, 6) et signal_var (0.1, 1.0, 4.0) sur le
# MEME schema de quantification (signe), observe-t-on une variation de d_ruzsa ?
# Si oui, c'est le signal que l'organ attrape bien une propriete de l'objet ;
# sinon, c'est une confirmation du caractere "mapping affine".

import numpy as np
from ict.factor_geometry import make_factor_bases, synthesize_activations


def d_ruzsa_on_activations(n_factors, signal_var, noise_var=0.1, seed=0, coord=0):
    rng = np.random.default_rng(seed)
    bases, _ = make_factor_bases(
        n_factors=n_factors, dim=24, mode='orthogonal', rng=rng)
    X_act, _ = synthesize_activations(
        bases, n_tokens=4000, signal_var=signal_var, noise_var=noise_var, rng=rng)
    X_q = sign_quantize(X_act)
    marg = marginal_distribution_n(X_q, coord=coord, n_bins=2)
    return marg, d_ruzsa(marg, marg, 2)


print("Variation de n_factors (signal_var=1.0 fixe) :")
for nf in [1, 3, 6]:
    marg, d = d_ruzsa_on_activations(n_factors=nf, signal_var=1.0)
    print(f"  n_factors={nf} : marg={marg}  d_ruzsa={d:.6f}")

print("\nVariation de signal_var (n_factors=3 fixe) :")
for sv in [0.1, 1.0, 4.0]:
    marg, d = d_ruzsa_on_activations(n_factors=3, signal_var=sv)
    print(f"  signal_var={sv:.1f} : marg={marg}  d_ruzsa={d:.6f}")
print("\n(attendu : la mesure depend des deux, mais par quel mecanisme ?)")


Variation de n_factors (signal_var=1.0 fixe) :
  n_factors=1 : marg={0: 0.5035, 1: 0.4965}  d_ruzsa=0.000035
  n_factors=3 : marg={0: 0.5045, 1: 0.4955}  d_ruzsa=0.000058
  n_factors=6 : marg={0: 0.478, 1: 0.522}  d_ruzsa=0.001394

Variation de signal_var (n_factors=3 fixe) :
  signal_var=0.1 : marg={0: 0.50075, 1: 0.49925}  d_ruzsa=0.000002
  signal_var=1.0 : marg={0: 0.5045, 1: 0.4955}  d_ruzsa=0.000058
  signal_var=4.0 : marg={0: 0.50525, 1: 0.49475}  d_ruzsa=0.000080

(attendu : la mesure depend des deux, mais par quel mecanisme ?)


In [7]:
# Exemple guide 3 -- d_ruzsa sur un objet ICT different : les bases elles-memes.
# (Exemple guide resolu : la solution complete est conservee ci-dessous.)
# Au lieu des activations generees, on prend les colonnes des sous-espaces
# factoriels et on les quantifie. La structure est differente (les colonnes
# sont des vecteurs de R^24, pas des activations) -- le temoin de la section
# precedente s'applique-t-il ici aussi ?

import numpy as np
from ict.factor_geometry import make_factor_bases


def basis_marginal_distribution(B, n_bins=2, coord=0):
    """Distribution marginale des coefficients d'une base B ∈ R^{D × k}."""
    return marginal_distribution_n(sign_quantize(B.T), coord=coord, n_bins=n_bins)


rng = np.random.default_rng(0)
bases, _ = make_factor_bases(n_factors=3, dim=24, mode='orthogonal', rng=rng)
print("Bases orthogonales (3 sous-espaces de dim 8) :")
for i, B in enumerate(bases):
    marg = basis_marginal_distribution(B, n_bins=2, coord=0)
    d = d_ruzsa(marg, marg, 2)
    print(f"  Base {i} (shape={B.shape}) : marg={marg}  d_ruzsa={d:.6f}")

print("\nComparaison avec les activations de la cellule 6 (meme seed=0) :")
print("  activations, coord 0 : d_ruzsa = voir cellule 6 (marge sign)")
print("  bases,    coord 0 : d_ruzsa varie selon i (les bases sont des")
print("                          vecteurs de R^24, leur distribution est differente")
print("                          de celle des activations generees).")


Bases orthogonales (3 sous-espaces de dim 8) :
  Base 0 (shape=(24, 8)) : marg={0: 0.75, 1: 0.25}  d_ruzsa=0.143156
  Base 1 (shape=(24, 8)) : marg={0: 0.375, 1: 0.625}  d_ruzsa=0.042746
  Base 2 (shape=(24, 8)) : marg={0: 0.625, 1: 0.375}  d_ruzsa=0.042746

Comparaison avec les activations de la cellule 6 (meme seed=0) :
  activations, coord 0 : d_ruzsa = voir cellule 6 (marge sign)
  bases,    coord 0 : d_ruzsa varie selon i (les bases sont des
                          vecteurs de R^24, leur distribution est differente
                          de celle des activations generees).


## 6 bis. Temoin de structure : les activations portent-elles la factorisation ?

Les cellules precedentes mesurent `d_ruzsa` sur l'objet ICT. Une question les precede : **que porte l'objet ?** Les activites synthetisees par `ict/factor_geometry.py` sont-elles encore factorisees, une fois qu'on ne les observe qu'a travers leurs statistiques ?

Le generateur construit `dim_per_factor = dim // n_factors` sous-espaces **orthonormes**. En regime `orthogonal`, les trois blocs de dimension 8 de ce carnet forment une base orthonormee **complete** de R^24 : les activites sont alors `X = W . Z`, avec `W` orthogonale et `Z` a composantes gaussiennes independantes -- c'est-a-dire une **gaussienne isotrope**, dont la factorisation est entierement absorbee par la rotation `W`. Une statistique invariante par rotation ne peut alors plus distinguer une decomposition en 3 x 8 d'une autre, et les statistiques axiales (une marginale) d'une gaussienne isotrope sont symetriques et quasi uniformes, quelle que soit la direction choisie.

Ce temoin le mesure par la **methode de la bande nulle** (celle d'ICT-36) : l'etendue du spectre empirique de la covariance des activites est comparee a celle de `K` gaussiennes isotropes de meme forme et de meme variance. Le regime `superposed` sert de **controle** : la ou les sous-espaces se recouvrent, la structure doit redevenir visible.


In [8]:
# Temoin de STRUCTURE : les activations de l'objet ICT portent-elles la
# factorisation ? Comparaison a une BANDE NULLE (meme methode que la bande
# nulle d'ICT-36) : l'etendue du spectre empirique de la covariance des
# activations est comparee a celle de K gaussiennes isotropes de meme forme
# et de meme variance.
# Regime ORTHOGONAL : la configuration de ce carnet (3 x 8 dans R^24).
# Regime SUPERPOSE (overlap = 0.5) : CONTROLE -- la ou les sous-espaces se
# recouvrent, la structure doit redevenir visible.
import sys
import os
import numpy as np

# Le module ict/ est dans le meme dossier que le carnet (cf cellule 6).
sys.path.insert(0, os.path.abspath('..'))
from ict.factor_geometry import (
    make_factor_bases,
    synthesize_activations,
    weighted_pca,
)

N_TOKENS, DIM, N_FACTORS = 4000, 24, 3
SIGNAL_VAR, NOISE_VAR = 1.0, 0.1
K_NULL = 20


def etendue_spectre(X):
    """Etendue (max - min) des valeurs propres de la covariance empirique.

    L'ORGANE du module est utilise tel quel : `weighted_pca` rend les valeurs
    singulieres de la covariance ponderee centree (poids uniformes), donc les
    valeurs propres en sont les carres divises par N. On ne recalcule pas la
    covariance ici.
    """
    _, _, sv, _ = weighted_pca(X)
    ev = (sv ** 2) / X.shape[0]
    return float(ev.max() - ev.min())


# --- Bande nulle : gaussienne isotrope de meme variance (signal + bruit) ---
var_totale = SIGNAL_VAR + NOISE_VAR
rng_null = np.random.default_rng(1234)
etendues_null = [etendue_spectre(
    rng_null.standard_normal((N_TOKENS, DIM)) * np.sqrt(var_totale))
    for _ in range(K_NULL)]
bande = (min(etendues_null), max(etendues_null))
print(f"Bande nulle (K={K_NULL}, gaussienne isotrope N(0, {var_totale} I)) :")
print(f"  etendue min={bande[0]:.6f}   etendue max={bande[1]:.6f}")

# --- Regime ORTHOGONAL : la configuration de ce carnet ---
rng = np.random.default_rng(0)
bases, dim_par_facteur = make_factor_bases(
    n_factors=N_FACTORS, dim=DIM, mode='orthogonal', rng=rng)
X, _ = synthesize_activations(
    bases, n_tokens=N_TOKENS, signal_var=SIGNAL_VAR, noise_var=NOISE_VAR, rng=rng)
e_orth = etendue_spectre(X)
verdict_orth = ("DANS la bande nulle" if bande[0] <= e_orth <= bande[1]
                else "HORS de la bande nulle")
print(f"\nRegime orthogonal ({N_FACTORS} x {dim_par_facteur} dans R^{DIM}, "
      f"config de ce carnet) :")
print(f"  etendue du spectre = {e_orth:.6f}  -> {verdict_orth}")

# --- Controle : regime SUPERPOSE (overlap = 0.5) ---
rng_sup = np.random.default_rng(0)
bases_sup, _ = make_factor_bases(
    n_factors=N_FACTORS, dim=DIM, mode='superposed', overlap=0.5, rng=rng_sup)
X_sup, _ = synthesize_activations(
    bases_sup, n_tokens=N_TOKENS, signal_var=SIGNAL_VAR, noise_var=NOISE_VAR,
    rng=rng_sup)
e_sup = etendue_spectre(X_sup)
verdict_sup = ("DANS la bande nulle" if bande[0] <= e_sup <= bande[1]
                else "HORS de la bande nulle")
print(f"\nControle, regime superpose (overlap = 0.5) :")
print(f"  etendue du spectre = {e_sup:.6f}  -> {verdict_sup}")
print(f"  rapport a la borne haute de la bande = {e_sup / bande[1]:.2f}")


Bande nulle (K=20, gaussienne isotrope N(0, 1.1 I)) :
  etendue min=0.276650   etendue max=0.361326

Regime orthogonal (3 x 8 dans R^24, config de ce carnet) :
  etendue du spectre = 0.331660  -> DANS la bande nulle

Controle, regime superpose (overlap = 0.5) :
  etendue du spectre = 1.281399  -> HORS de la bande nulle
  rapport a la borne haute de la bande = 3.55


### Lecture du resultat

Trois mesures, trois verdicts :

- **Bande nulle** : `K = 20` gaussiennes isotropes de meme variance (`N(0, 1.1 I)`) donnent une etendue de spectre empirique comprise entre **0.276650** et **0.361326** : c'est la bande de fluctuation d'un objet **sans** structure factorisee.
- **Regime orthogonal** (la configuration de ce carnet : 3 sous-espaces de dimension 8 dans R^24) : etendue = **0.331660** -> **DANS** la bande nulle. Sur le critere mesure, les activations sont dans la bande d'une gaussienne isotrope : leur second ordre **ne porte pas** la factorisation. C'est la consequence directe du fait que les trois blocs forment une base orthonormee **complete** de R^24 -- la factorisation est absorbee par la rotation.
- **Controle, regime superpose** (overlap = 0.5) : etendue = **1.281399**, soit **3.55 fois** la borne haute de la bande -> **HORS** de la bande nulle. Le temoin n'est donc pas une tautologie : il distingue le regime ou la structure est visible de celui ou elle ne l'est pas.

**Ce que cela dit de la lecture de la section 4** : le `d_ruzsa(X;X) = 0.000058` bits de la marginale sign(Z/2Z) n'est pas un verdict sur la geometrie factorisee. Il mesure une marginale **quasi equilibree** (`{0: 0.5045, 1: 0.4955}`) -- consequence directe de l'isotropie de l'objet, non l'absence de structure. Un instrument scalaire pose sur une marginale ne peut pas lire une factorisation qui n'est pas dans les statistiques marginales.

**L'instrument qui la lit** : les outils de **sous-espace** du meme module -- `max_principal_angle`, `weighted_pca`, `nc_at` -- qui mesurent l'angle entre sous-espaces et non l'etendue d'un spectre marginal. C'est la lecon du carnet : le bon outil depend de ce que l'objet **porte**, pas de la capacite qu'on aimerait lui trouver.


## 7. Verdict -- organ-first et statut du contrat

**5 questions d'organ-first** ([organ-first-implementation.md](../../../.claude/rules/organ-first-implementation.md)) :

1. **Quelle serie possede deja la semantique de cette operation ?** -- La serie **SymbolicAI / Lean / ANALYSE** (digestion PFR-Tao), carnet `ANALYSE-04-PFR-Primitives-Python.ipynb`, cellule 2. L'organ est deja present, code + tests executes.
2. **Peut-on invoquer son module / librairie / service reel ?** -- **Pas directement** : le carnet expose la fonction en cellule, pas en module. Vendoring verbatim (cf cellule 3 de ce carnet), avec attribution SPDX-License-Identifier GPL-2.0-or-later.
3. **Sinon, que faut-il exporter ou refactorer dans la serie source pour la rendre consommable ?** -- Un refactor cote source (extraire `d_ruzsa` et `entropy_bits` dans `symbolicai/lean/analyse/ruzsa.py`) rendrait la greffe triviale. **Suivi propose** : issue ou PR sur la serie source, hors perimetre de ce carnet. Le present test de limite est un cas d'usage, pas un refactor.
4. **Quel temoin negatif l'organe natif fournit-il ?** -- Le carnet organ expose deja un test de limite cellule 4 : (a) deux lois de groupe differentes donnent deux valeurs differentes pour les memes distributions ; (b) sans inversibilite de `x ⇒ x - y`, `d` devient negatif. Le present carnet **ajoute** un troisieme temoin : (c) l'overclaim de la cellule 1 (`d[X;X] = 0`) sur des distributions non uniformes sur G.
5. **Quelle autre serie assure la verification independante du resultat ?** -- La serie **SymbolicAI / Lean / ANALYSE** : le carnet organ porte ses propres tests de limite (cellules 2 et 4), et la discussion [#18405](https://github.com/jsboige/CoursIA/issues/18405) documente une reproduction independante du meme overclaim sur Z/4Z.

**Statut du contrat corrige** : `d[X;Y] = 0` si et seulement si Y = X + a (translation invariante). L'enonce de la cellule 1 vaut pour X uniforme sur G, cas particulier. L'overclaim est **documente** dans la discussion [#18405](https://github.com/jsboige/CoursIA/issues/18405).

**Statut op 3 de la table #12204** : reste en **file d'attente** (1 attestation locale ANALYSE-04 + 1 attestation pedagogique non-independante Infer-20). Le present carnet **ne promeut pas** l'op 3 -- il documente un test de limite, qui est l'un des deux aboutissants acceptables par #18405 (point 3 du A faire : "Distance de Ruzsa : application sur un objet ICT a structure de groupe, ou test de limite documente"). Le temoin negatif delivre (cellule 8) est exactement le deuxieme aboutissant.


## À retenir

*La distance de Ruzsa lit la geometrie factorisee d'ICT-36, mais l'organe qui la porte est le carnet ANALYSE-04-PFR-Primitives (serie SymbolicAI/Lean/ANALYSE), importe ici verbatim avec son attribution -- et ce carnet y ajoute un troisieme temoin negatif : l'identite `d[X;X] = 0` ne tient que sur les distributions uniformes sur G.*

- Les deux temoins de l'organe restent valables : deux lois de groupe differentes donnent deux valeurs differentes pour les memes distributions ; sans inversibilite de `x -> x - y`, la distance devient negative.
- **La mesure dépend du plongement, pas seulement de l'objet** : un même jeu d'activations rend `d_ruzsa` = 0.000058 (signe, Z/2Z), 0.000000 (3- et 5-quantiles) et 0.064937 (hash LSB) selon le schéma de quantification -- c'est le « mapping affine » que l'organ source dénonçait déjà. Le schéma hash est le plus radical : il lit le **dernier bit de la mantisse float64**, sa valeur n'est donc pas reproductible d'une machine à l'autre (0.072061 sur la pile de la version initiale, 0.064937 sur celle de l'exécution committée ici) -- c'est la même leçon, poussée à sa limite.
- **Ce que l'objet porte, et ce qu'il ne porte pas** : en régime orthogonal, les trois sous-espaces de dimension 8 pavent R^24 -- les activations sont une gaussienne isotrope dont la factorisation est absorbée par la rotation. Leur second ordre ne porte donc pas la factorisation : l'étendue de leur spectre empirique tombe dans la bande nulle d'une gaussienne isotrope, là où le régime superposé la rend visible (3.55 fois la borne haute de la bande). La valeur quasi nulle que `d_ruzsa` rend sur la marginale quantifiée mesure donc l'isotropie de l'objet, non l'absence de structure -- l'instrument qui lit la factorisation est celui des sous-espaces (`max_principal_angle`, `weighted_pca`, `nc_at`).
- **L'invariance exacte est la translation commune** : `d[X+a; Y+a] = d[X;Y]`, et non `d[X; X+a] = 0` -- le Cas 3 le tient par la mesure (`d[X;X]`, `d[X;X+1]`, `d[X;X+2]` tous à 0.5 sur Z/4Z).
- Le suivi proposé : extraire `d_ruzsa` et `entropy_bits` dans `symbolicai/lean/analyse/ruzsa.py` rendrait la greffe triviale -- issue ou PR sur la serie source, hors perimetre de ce carnet.

## Vérifiez votre compréhension

1. Quel est le troisième témoin négatif que ce carnet ajoute à l'organe importé ?
2. D'où vient l'organe qui porte `d_ruzsa`, et que valent ses deux témoins ?
3. Quel suivi est proposé, et pourquoi hors périmètre ?
4. Que porte l'objet ICT de ses statistiques du second ordre, et pourquoi sa marginale quantifiée est-elle si équilibrée ?

<details>
<summary>Réponses</summary>

1. L'identité `d[X;X] = 0` **ne tient que sur les distributions uniformes sur G** — en dehors, la « distance » d'une distribution à elle-même n'est pas nulle.
2. Il vient du carnet **ANALYSE-04-PFR-Primitives** (série SymbolicAI/Lean/ANALYSE), importé verbatim avec son attribution. Ses deux témoins : deux lois de groupe différentes donnent deux valeurs différentes pour les mêmes distributions ; sans inversibilité de `x -> x - y`, la distance devient négative.
3. Extraire `d_ruzsa` et `entropy_bits` dans `symbolicai/lean/analyse/ruzsa.py` rendrait la greffe triviale — mais c'est une issue ou PR sur la **série source**, hors périmètre de ce carnet.
4. En régime orthogonal, les trois sous-espaces de dimension 8 forment une base orthonormée **complète** de R^24 : les activations sont `X = W . Z` avec `W` orthogonale et `Z` gaussienne isotrope — la factorisation est absorbée par la rotation. L'étendue de leur spectre empirique tombe donc dans la bande nulle d'une gaussienne isotrope (0.331660 contre une bande [0.276650, 0.361326]), là où le régime superposé la dépasse (1.281399). Une marginale d'une gaussienne isotrope est symétrique et quasi équilibrée, d'où un `d_ruzsa(X;X)` quasi nul : cette mesure est celle de l'isotropie, pas d'une absence de structure.

</details>

***

Trois exercices non resolus, a completer. Les trois cellules precedentes (« Exemple guide 1 » a « Exemple guide 3 ») resolvent des variantes du meme objet : les garder sous les yeux pour s'en inspirer, mais **aucune n'est a recopier** -- chaque exercice mesure quelque chose de **neuf**.

Regle C.1 : les stubs s'executent sans erreur (`print("Exercice a completer")`, `result = None  # TODO etudiant`), aucun `raise`.


### Exercice 1 -- d_ruzsa sur la jointure de deux coordonnees

Les « Exemple guide » appliquent `d_ruzsa` a une **marginale** (une seule coordonnee quantifiee). Ici, mesurer la **jointure** de deux coordonnees `(x_i, x_j)`.

**Objectif** : comparer `d_ruzsa` sur la loi jointe de deux coordonnees quantifiees a sa valeur sur la marginale seule. La dependance entre coordonnees fait-elle varier la mesure, ou la marginale suffit-elle ?

**Etape 1** : reprendre les activations quantifiees par signe (`X_q1`, groupe `Z/2Z`), disponibles plus haut.
**Etape 2** : former l'index joint `idx = X_q1[:, i] * 2 + X_q1[:, j]` (groupe `Z/4Z`, 4 etats).
**Etape 3** : en deduire la distribution `{idx: proba}`, puis `d_ruzsa(joint, joint, 4)`.
**Etape 4** : comparer a la valeur sur la marginale (Exemple guide 1) et conclure sur le role de la dependance entre coordonnees.


In [9]:
# Exercice 1 -- d_ruzsa sur la JOINTURE de deux coordonnees (i, j).
# Les exemples precedents utilisent une MARGINALE ; ici on joint i et j.
# ETAPE 1 : X_q1 (activations quantifiees par signe, Z/2Z) est disponible plus haut.
# ETAPE 2 : idx = X_q1[:, i] * 2 + X_q1[:, j]  -> groupe Z/4Z (4 etats).
# ETAPE 3 : distribution jointe via np.bincount(idx, minlength=4), puis
#           d_ruzsa(joint, joint, 4).
# ETAPE 4 : comparer a la marginale (Exemple guide 1) et conclure sur le role
#           de la dependance entre les deux coordonnees.
result_joint_ex1 = None  # TODO etudiant
print("Exercice a completer")


Exercice a completer


### Exercice 2 -- la sensibilite au decalage avant quantification

La section 1 etablit l'invariant `d[X + a; Y + a] = d[X; Y]` (translation **commune**) : la distance de Ruzsa ne change pas quand on translate les deux lois du meme `a`. La lecture du temoin de limite (Cas 3) precise la portee exacte : `d[X + a; Y + a] = d[X; Y]`, et **non** `d[X; X + a] = 0`.

**Objectif** : mesurer la **sensibilite de l'objet ICT a un decalage applique avant la quantification**, et non eprouver la translation **commune** des deux lois. Un decalage `a` est applique a la coordonnee **avant** la quantification par signe ; on compare ensuite la marginale decalee `marg_a` a la marginale **originale** `marg1` (schema signe `Z/2Z`, definie plus haut). Une seule des deux marginales est decalee : l'invariant de groupe `d[X + a; Y + a] = d[X; Y]` ne s'applique donc pas ici, et la valeur de reference en `a = 0` n'est pas 0 mais l'auto-distance `d1` de `marg1` (mesuree `0.000058`, cf. temoin de limite) -- en `a = 0` les deux marginales coincident, donc `d_ruzsa(marg_a, marg1, 2) = d1`. L'ecart a `d1` mesure la sensibilite au decalage ; il depend de `a`. **Un decalage avant quantification n'est pas une translation dans le groupe** : `sign(X + a)` n'est pas `sign(X) + a` -- le seuil de signe n'est pas un deplacement de groupe.

**Etape 1** : reprendre les activations brutes `X` (seed 0, shape `4000 x 24`).
**Etape 2** : pour `a` dans `[-0.5, -0.1, 0.0, 0.1, 0.5]`, quantifier `sign_quantize(X + a)` et prendre la marginale de la coordonnee 0 -> `marg_a`.
**Etape 3** : calculer `d_ruzsa(marg_a, marg1, 2)` pour chaque `a` (decale contre **original** ; a comparer a `d1`, l'auto-distance de `marg1`, atteinte en `a = 0` ou les deux marginales coincident).
**Etape 4** : rassembler les valeurs et conclure -- la valeur depend-elle de `a` ? Qu'est-ce que cela dit du plongement (un seuil de signe, et non un deplacement de groupe) ?


In [10]:
# Exercice 2 -- sensibilite au decalage avant quantification.
# Invariant (section 1) : d[X+a; Y+a] = d[X; Y] (translation COMMUNE), et NON
# d[X; X+a] = 0 (cf la lecture du temoin de limite, Cas 3).
# Ici `a` est un decalage applique a la coordonnee AVANT la quantification par
# signe : un decalage avant quantification n'est PAS une translation de groupe
# (sign(X + a) n'est pas sign(X) + a -- le seuil de signe n'est pas un
# deplacement de groupe).
# ETAPE 1 : X (activations brutes, seed 0, shape 4000 x 24) est disponible plus haut.
# ETAPE 2 : pour a in [-0.5, -0.1, 0.0, 0.1, 0.5] :
#             marg_a = marginal_distribution_n(sign_quantize(X + a), coord=0, n_bins=2)
# ETAPE 3 : d_a = d_ruzsa(marg_a, marg1, 2)  -- decale contre ORIGINAL
#             (marg1 = marginale signe de X, definie plus haut ; d1 = son
#             auto-distance, atteinte en a = 0) ; a comparer a d1.
# ETAPE 4 : rassembler les valeurs (a -> d_ruzsa) et conclure sur la
#             sensibilite au decalage (les d_a contre d1) -- une seule marginale
#             est decalee, donc l'invariant de translation COMMUNE de la section 1
#             ne s'applique pas ici.
result_translation_ex2 = None  # TODO etudiant
print("Exercice a completer")


Exercice a completer


### Exercice 3 -- balayage de la profondeur de quantification

Le temoin de limite (cellule 8) compare trois **schemas** sur un objet unique. Ici, balayer la **profondeur** `n_bins` d'un meme schema (`quantile_quantize`) et suivre l'evolution de `d_ruzsa`.

**Objectif** : mesurer `d_ruzsa` pour `n_bins` dans `{2, 4, 8, 16, 32}` sur la meme coordonnee. La valeur tend-elle vers 0 quand la resolution augmente, ou persiste-t-elle ?

**Etape 1** : reprendre les activations brutes `X` (seed 0).
**Etape 2** : pour chaque `n_bins`, appliquer `quantile_quantize(X, n_bins)` puis prendre la marginale de la coordonnee 0.
**Etape 3** : calculer `d_ruzsa(marg, marg, n_bins)` pour chaque `n_bins`.
**Etape 4** : presenter la suite `n_bins -> d_ruzsa` et conclure -- la « signature du plongement » s'attenue-t-elle avec la finesse, ou reste-t-elle ?


In [11]:
# Exercice 3 -- balayage de la profondeur de quantification.
# Pour n_bins dans {2, 4, 8, 16, 32} : quantifier par quantile_quantize, prendre
# la marginale de la coordonnee 0, puis calculer d_ruzsa(marg, marg, n_bins).
# Remplir result_sweep_ex3 avec la liste [(n_bins, d_ruzsa_value), ...].

result_sweep_ex3 = None  # TODO etudiant
print("Exercice a completer")


Exercice a completer


## Pour aller plus loin

- **Source organ** : [ANALYSE-04-PFR-Primitives-Python.ipynb](../../SymbolicAI/Lean/ANALYSE/ANALYSE-04-PFR-Primitives-Python.ipynb) (cellules 1, 2, 4, 5).
- **Module importe** : [ict/factor_geometry.py](ict/factor_geometry.py) (PR #15514).
- **Issue de coordination** : [#18405](https://github.com/jsboige/CoursIA/issues/18405) (greffe PFR / op 3 table #12204).
- **Verification independante** : reproduction du meme overclaim dans la discussion [#18405](https://github.com/jsboige/CoursIA/issues/18405).
- **Regle organ-first** : [organ-first-implementation.md](../../../.claude/rules/organ-first-implementation.md) (5 questions).
- **[ICT-36c — FLens, pont Higuchi](ICT-36c-FLens-HiguchiBridge-Python.ipynb)** : le pont vers la regularite temporelle -- la dimension de Higuchi croise les instruments geometriques de la famille FLens.
- [ICT-36e -- FLens-SNR](ICT-36e-FLens-SNR-Python.ipynb) : cette référence au plancher de bruit, construite et mesurée.
- [ICT-36 -- FactoredGeometry](ICT-36-FLens-FactoredGeometry-Python.ipynb) : le carnet parent, source du module `ict/factor_geometry.py`.
- **[ICT-0 — Cadrage de la serie](ICT-0-Framing.md)** : le principe directeur de la serie ICT.